# Engine B SFT Notebook

End-to-end preparation and supervised fine-tuning workflow for Engine B.

In [13]:
import os
import json
from pathlib import Path

## 1. Paths and Configuration

Define project roots and dataset locations.

In [14]:
# Paths / config

from pathlib import Path

# Notebook location:
# gruve-ai-expo/engine_b_qwen/notebooks/engine_b_sft.ipynb

PROJECT_ROOT = Path.cwd().parent.parent

TRAIN_OCR_DIR = PROJECT_ROOT / "dataset" / "train" / "ocr"
TRAIN_GT_DIR  = PROJECT_ROOT / "dataset" / "train" / "entities"

TEST_OCR_DIR  = PROJECT_ROOT / "dataset" / "test" / "ocr"
TEST_GT_DIR   = PROJECT_ROOT / "dataset" / "test" / "entities"

OUTPUT_DIR = PROJECT_ROOT / "engine_b_qwen" / "outputs_engine_b"
NOTEBOOK_DIR = Path.cwd()

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_OCR_DIR exists:", TRAIN_OCR_DIR.exists(), TRAIN_OCR_DIR)
print("TRAIN_GT_DIR exists :", TRAIN_GT_DIR.exists(), TRAIN_GT_DIR)
print("TEST_OCR_DIR exists :", TEST_OCR_DIR.exists(), TEST_OCR_DIR)
print("TEST_GT_DIR exists  :", TEST_GT_DIR.exists(), TEST_GT_DIR)
print("OUTPUT_DIR exists   :", OUTPUT_DIR.exists(), OUTPUT_DIR)

PROJECT_ROOT: /Users/jkim/Projects/gruve-ai-expo
TRAIN_OCR_DIR exists: True /Users/jkim/Projects/gruve-ai-expo/dataset/train/ocr
TRAIN_GT_DIR exists : True /Users/jkim/Projects/gruve-ai-expo/dataset/train/entities
TEST_OCR_DIR exists : True /Users/jkim/Projects/gruve-ai-expo/dataset/test/ocr
TEST_GT_DIR exists  : True /Users/jkim/Projects/gruve-ai-expo/dataset/test/entities
OUTPUT_DIR exists   : True /Users/jkim/Projects/gruve-ai-expo/engine_b_qwen/outputs_engine_b


## 2. Dataset Sanity Checks

Verify OCR and entity files line up by document ID.

In [15]:
# sanity check - list files and match IDs
train_ocr_files = sorted(TRAIN_OCR_DIR.glob("*.txt"))
train_gt_files = sorted(TRAIN_GT_DIR.glob("*.txt"))

test_ocr_files = sorted(TEST_OCR_DIR.glob("*.txt"))
test_gt_files = sorted(TEST_GT_DIR.glob("*.txt"))

print(f"Train OCR files: {len(train_ocr_files)}")
print(f"Train GT files : {len(train_gt_files)}")
print(f"Test OCR files : {len(test_ocr_files)}")
print(f"Test GT files  : {len(test_gt_files)}")

train_ocr_ids = {f.stem for f in train_ocr_files}
train_gt_ids = {f.stem for f in train_gt_files}
test_ocr_ids = {f.stem for f in test_ocr_files}
test_gt_ids = {f.stem for f in test_gt_files}

train_common = sorted(train_ocr_ids & train_gt_ids)
test_common = sorted(test_ocr_ids & test_gt_ids)

print(f"\nMatched train pairs: {len(train_common)}")
print(f"Matched test pairs : {len(test_common)}")

print("\nFirst 10 matched train IDs:")
for doc_id in train_common[:10]:
    print("-", doc_id)

print("\nFirst 10 matched test IDs:")
for doc_id in test_common[:10]:
    print("-", doc_id)

Train OCR files: 626
Train GT files : 626
Test OCR files : 347
Test GT files  : 347

Matched train pairs: 626
Matched test pairs : 347

First 10 matched train IDs:
- X00016469612
- X00016469619
- X00016469620
- X00016469622
- X00016469623
- X00016469669
- X00016469672
- X00016469676
- X51005200938
- X51005230617

First 10 matched test IDs:
- X00016469670
- X00016469671
- X51005200931
- X51005230605
- X51005230616
- X51005230621
- X51005230648
- X51005230657
- X51005230659
- X51005268275


## 3. Single Example Inspection

Load one matched training sample to inspect raw inputs.

In [16]:
# Load one matched train example and inspect it

sample_id = train_common[0]

ocr_path = TRAIN_OCR_DIR / f"{sample_id}.txt"
gt_path = TRAIN_GT_DIR / f"{sample_id}.txt"

with open(ocr_path, "r", encoding="utf-8") as f:
    ocr_text = f.read()

with open(gt_path, "r", encoding="utf-8") as f:
    gt_text = f.read()

print("Sample ID:", sample_id)
print("\n=== OCR TEXT ===\n")
print(ocr_text[:3000])  # truncate if huge

print("\n=== GROUND TRUTH RAW ===\n")
print(gt_text[:3000])

Sample ID: X00016469612

=== OCR TEXT ===

tan woon yann

BOOK TA-K (TAMAN DAYA) SDN BHD
B94 7-W
NO.5? 55,57 & 59, JALAN SAGU 18,
TAMAN DAYA
81100 JOHOR BAHRU,
JOHOR.

LAM MCU

Document Ho : TDO1167104

 

Date 25/12/2018 8:13:39 PM
Cashier MANIS
Member
CASH BILL
CODE/DESC PRICE — Disc AMOUIT
Quy RM RM
9556939040118 KF MODELLING CLAY KIDDY FISH
1PC * 9.00) 6,00 9.00
Total : 9.00
Rour ding Adjustment 0.00

Round: :d Total (RM):

9.60

Cash
CHANGE

  

GOODS SOLD ARE NOT RETURNAR
EXCHANGEABLE

 

THANK YOU.
PLEASE COME AGATY t


=== GROUND TRUTH RAW ===

{
    "company": "BOOK TA .K (TAMAN DAYA) SDN BHD",
    "date": "25/12/2018",
    "address": "NO.53 55,57 & 59, JALAN SAGU 18, TAMAN DAYA, 81100 JOHOR BAHRU, JOHOR.",
    "total": "9.00"
}


## 4. Formatting Helpers

Build training targets and message-format utilities.

In [17]:
def load_ground_truth(gt_path: Path):
    with open(gt_path, "r", encoding="utf-8") as f:
        return json.load(f)


def build_target_json(doc_id: str, gt: dict, engine_name: str = "qwen-engine-b-sft"):
    return {
        "engine": engine_name,
        "source_file": f"{doc_id}.txt",
        "extracted_fields": {
            "company": gt.get("company"),
            "date": gt.get("date"),
            "address": gt.get("address"),
            "total": gt.get("total"),
        },
    }


gt_dict = load_ground_truth(gt_path)
target_json = build_target_json(sample_id, gt_dict)

print("=== PARSED GROUND TRUTH DICT ===\n")
print(json.dumps(gt_dict, indent=2, ensure_ascii=False))

print("\n=== FINAL TARGET JSON ===\n")
print(json.dumps(target_json, indent=2, ensure_ascii=False))

=== PARSED GROUND TRUTH DICT ===

{
  "company": "BOOK TA .K (TAMAN DAYA) SDN BHD",
  "date": "25/12/2018",
  "address": "NO.53 55,57 & 59, JALAN SAGU 18, TAMAN DAYA, 81100 JOHOR BAHRU, JOHOR.",
  "total": "9.00"
}

=== FINAL TARGET JSON ===

{
  "engine": "qwen-engine-b-sft",
  "source_file": "X00016469612.txt",
  "extracted_fields": {
    "company": "BOOK TA .K (TAMAN DAYA) SDN BHD",
    "date": "25/12/2018",
    "address": "NO.53 55,57 & 59, JALAN SAGU 18, TAMAN DAYA, 81100 JOHOR BAHRU, JOHOR.",
    "total": "9.00"
  }
}


In [18]:
SYSTEM_PROMPT = """You are an information extraction system.

Extract structured information from OCR receipt text.
Return the result as a valid JSON object with the following structure:

{
  "engine": "...",
  "source_file": "...",
  "extracted_fields": {
    "company": "...",
    "date": "...",
    "address": "...",
    "total": "..."
  }
}

Only return valid JSON. Do not include explanations.
"""


def format_training_example(ocr_text: str, target_json: dict):

    user_prompt = f"""Extract the company, date, address, and total from the following OCR receipt text.

OCR TEXT:
{ocr_text}
"""

    assistant_output = json.dumps(target_json, indent=2, ensure_ascii=False)

    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": assistant_output},
        ]
    }

In [19]:
example = format_training_example(ocr_text, target_json)

print(json.dumps(example, indent=2, ensure_ascii=False)[:2000])

{
  "messages": [
    {
      "role": "system",
      "content": "You are an information extraction system.\n\nExtract structured information from OCR receipt text.\nReturn the result as a valid JSON object with the following structure:\n\n{\n  \"engine\": \"...\",\n  \"source_file\": \"...\",\n  \"extracted_fields\": {\n    \"company\": \"...\",\n    \"date\": \"...\",\n    \"address\": \"...\",\n    \"total\": \"...\"\n  }\n}\n\nOnly return valid JSON. Do not include explanations.\n"
    },
    {
      "role": "user",
      "content": "Extract the company, date, address, and total from the following OCR receipt text.\n\nOCR TEXT:\ntan woon yann\n\nBOOK TA-K (TAMAN DAYA) SDN BHD\nB94 7-W\nNO.5? 55,57 & 59, JALAN SAGU 18,\nTAMAN DAYA\n81100 JOHOR BAHRU,\nJOHOR.\n\nLAM MCU\n\nDocument Ho : TDO1167104\n\n \n\nDate 25/12/2018 8:13:39 PM\nCashier MANIS\nMember\nCASH BILL\nCODE/DESC PRICE — Disc AMOUIT\nQuy RM RM\n9556939040118 KF MODELLING CLAY KIDDY FISH\n1PC * 9.00) 6,00 9.00\nTotal : 9.

## 5. Build Training Examples

Convert all matched samples into chat-format training records.

In [20]:
training_examples = []

for doc_id in train_common:

    ocr_path = TRAIN_OCR_DIR / f"{doc_id}.txt"
    gt_path = TRAIN_GT_DIR / f"{doc_id}.txt"

    with open(ocr_path, "r", encoding="utf-8") as f:
        ocr_text = f.read()

    gt_dict = load_ground_truth(gt_path)

    target_json = build_target_json(doc_id, gt_dict)

    example = format_training_example(ocr_text, target_json)

    training_examples.append(example)

print("Training examples created:", len(training_examples))

Training examples created: 626


In [21]:
print(json.dumps(training_examples[0], indent=2)[:2000])

{
  "messages": [
    {
      "role": "system",
      "content": "You are an information extraction system.\n\nExtract structured information from OCR receipt text.\nReturn the result as a valid JSON object with the following structure:\n\n{\n  \"engine\": \"...\",\n  \"source_file\": \"...\",\n  \"extracted_fields\": {\n    \"company\": \"...\",\n    \"date\": \"...\",\n    \"address\": \"...\",\n    \"total\": \"...\"\n  }\n}\n\nOnly return valid JSON. Do not include explanations.\n"
    },
    {
      "role": "user",
      "content": "Extract the company, date, address, and total from the following OCR receipt text.\n\nOCR TEXT:\ntan woon yann\n\nBOOK TA-K (TAMAN DAYA) SDN BHD\nB94 7-W\nNO.5? 55,57 & 59, JALAN SAGU 18,\nTAMAN DAYA\n81100 JOHOR BAHRU,\nJOHOR.\n\nLAM MCU\n\nDocument Ho : TDO1167104\n\n \n\nDate 25/12/2018 8:13:39 PM\nCashier MANIS\nMember\nCASH BILL\nCODE/DESC PRICE \u2014 Disc AMOUIT\nQuy RM RM\n9556939040118 KF MODELLING CLAY KIDDY FISH\n1PC * 9.00) 6,00 9.00\nTotal

## 6. Dataset Packaging

Install and prepare Hugging Face `Dataset` objects.

In [23]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 935.3 kB/s eta 0:00:00:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 1.2 MB/s eta 0:00:00-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 934.0 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.2/34.2 MB 627.2 kB/s eta 0:00:0000:0100:02
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 19.0.0
    Uninstalling pyarrow-19.0.0:
      Successfully uninstalled pyarrow-19.0.0
  Attempting uninstall: dill━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/7 [pyarrow]
    Found existing installation: dill 0.3.8━━━━━━━━━━━━━━━━━━━ 1/7 [pyarrow]
    Uninstalling dill-0.3.8:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/7 [pyarrow]
      Successfully uninstalled dill-0.3.8━━━━━━━━━━━━━━━━━━━━━ 1/7 [pyarrow]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [datasets]6/7 [datasets]ce-hub]


In [24]:
from datasets import Dataset

train_dataset = Dataset.from_list(training_examples)

print(train_dataset)

Dataset({
    features: ['messages'],
    num_rows: 626
})


## 7. Training Dependencies

Install libraries required for QLoRA/SFT training.

In [25]:
!pip install unsloth datasets transformers trl peft accelerate bitsandbytes

  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 710.5 kB/s eta 0:00:0000:0100:01
Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl (3.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 843.4 kB/s eta 0:00:00a 0:00:01
Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl (447 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 897.1 kB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 1.0 MB/s eta 0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 1.1 MB/s eta 0:00:00a 0:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

**The following cells require a supported GPU environment (e.g., Northeastern Explorer cluster).
They may not run on a local Mac environment.**  
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 

## 8. GPU Training Setup

Load model/tooling and configure training on a supported GPU environment.

In [32]:
import torch
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer

HAS_GPU = torch.cuda.is_available()

print("CUDA available:", HAS_GPU)
if HAS_GPU:
    try:
        from unsloth import FastLanguageModel
        print("FastLanguageModel imported successfully.")
    except Exception as e:
        print("Unsloth import failed:", e)
        FastLanguageModel = None
else:
    print("Skipping Unsloth import locally; run model/training cells on HPC.")
    FastLanguageModel = None

CUDA available: False
Skipping Unsloth import locally; run model/training cells on HPC.


In [33]:
MODEL_ID = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
SFT_OUTPUT_DIR = PROJECT_ROOT / "engine_b_qwen" / "outputs_engine_b" / "qwen_sft"
SFT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("MODEL_ID:", MODEL_ID)
print("MAX_SEQ_LENGTH:", MAX_SEQ_LENGTH)
print("SFT_OUTPUT_DIR:", SFT_OUTPUT_DIR)

MODEL_ID: unsloth/Qwen2.5-7B-Instruct-bnb-4bit
MAX_SEQ_LENGTH: 2048
SFT_OUTPUT_DIR: /Users/jkim/Projects/gruve-ai-expo/engine_b_qwen/outputs_engine_b/qwen_sft


In [34]:
if FastLanguageModel is None:
    print("Model loading skipped. Run this cell on HPC with GPU support.")
else:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    print("Model and tokenizer loaded.")

Model loading skipped. Run this cell on HPC with GPU support.


In [35]:
if FastLanguageModel is None:
    print("LoRA setup skipped. Run this cell on HPC with GPU support.")
else:
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
    )
    print("LoRA adapters attached.")

LoRA setup skipped. Run this cell on HPC with GPU support.


## 9. Prompt/Data Preview

Inspect final message structure before tokenization and training.

In [36]:
print("Example message structure:\n")
print(json.dumps(training_examples[0], indent=2)[:2000])

print("\nDataset size:", len(training_examples))
print("Keys in one example:", training_examples[0].keys())
print("Keys in messages[0]:", training_examples[0]["messages"][0].keys())

Example message structure:

{
  "messages": [
    {
      "role": "system",
      "content": "You are an information extraction system.\n\nExtract structured information from OCR receipt text.\nReturn the result as a valid JSON object with the following structure:\n\n{\n  \"engine\": \"...\",\n  \"source_file\": \"...\",\n  \"extracted_fields\": {\n    \"company\": \"...\",\n    \"date\": \"...\",\n    \"address\": \"...\",\n    \"total\": \"...\"\n  }\n}\n\nOnly return valid JSON. Do not include explanations.\n"
    },
    {
      "role": "user",
      "content": "Extract the company, date, address, and total from the following OCR receipt text.\n\nOCR TEXT:\ntan woon yann\n\nBOOK TA-K (TAMAN DAYA) SDN BHD\nB94 7-W\nNO.5? 55,57 & 59, JALAN SAGU 18,\nTAMAN DAYA\n81100 JOHOR BAHRU,\nJOHOR.\n\nLAM MCU\n\nDocument Ho : TDO1167104\n\n \n\nDate 25/12/2018 8:13:39 PM\nCashier MANIS\nMember\nCASH BILL\nCODE/DESC PRICE \u2014 Disc AMOUIT\nQuy RM RM\n9556939040118 KF MODELLING CLAY KIDDY FISH\n

In [37]:
def render_messages_as_text(example):
    parts = []
    for msg in example["messages"]:
        role = msg["role"].strip().lower()
        content = msg["content"].strip()
        parts.append(f"<|{role}|>\n{content}")
    return {"text": "\n\n".join(parts)}

train_dataset_text = train_dataset.map(render_messages_as_text)

print(train_dataset_text)
print("\n=== Sample rendered training text ===\n")
print(train_dataset_text[0]["text"][:3000])

Map:   0%|          | 0/626 [00:00<?, ? examples/s]

Dataset({
    features: ['messages', 'text'],
    num_rows: 626
})

=== Sample rendered training text ===

<|system|>
You are an information extraction system.

Extract structured information from OCR receipt text.
Return the result as a valid JSON object with the following structure:

{
  "engine": "...",
  "source_file": "...",
  "extracted_fields": {
    "company": "...",
    "date": "...",
    "address": "...",
    "total": "..."
  }
}

Only return valid JSON. Do not include explanations.

<|user|>
Extract the company, date, address, and total from the following OCR receipt text.

OCR TEXT:
tan woon yann

BOOK TA-K (TAMAN DAYA) SDN BHD
B94 7-W
NO.5? 55,57 & 59, JALAN SAGU 18,
TAMAN DAYA
81100 JOHOR BAHRU,
JOHOR.

LAM MCU

Document Ho : TDO1167104

 

Date 25/12/2018 8:13:39 PM
Cashier MANIS
Member
CASH BILL
CODE/DESC PRICE — Disc AMOUIT
Quy RM RM
9556939040118 KF MODELLING CLAY KIDDY FISH
1PC * 9.00) 6,00 9.00
Total : 9.00
Rour ding Adjustment 0.00

Round: :d Total (RM):

9.60

Cas

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.


## 10. Tokenization Checks

Validate token counts and truncation behavior.

In [38]:
if "tokenizer" not in globals() or tokenizer is None:
    print("Tokenizer not loaded yet. Run tokenization checks on HPC after model/tokenizer load.")
else:
    sample_text = train_dataset_text[0]["text"]
    tokenized = tokenizer(sample_text, truncation=False)
    print("Sample token count:", len(tokenized["input_ids"]))

Tokenizer not loaded yet. Run tokenization checks on HPC after model/tokenizer load.


## 11. SFT Hyperparameters and Run

Set training arguments and launch supervised fine-tuning.

In [39]:
TRAIN_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
NUM_EPOCHS = 2
LEARNING_RATE = 2e-4
LOGGING_STEPS = 10
SAVE_STEPS = 100
WARMUP_STEPS = 10

print({
    "batch_size": TRAIN_BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "epochs": NUM_EPOCHS,
    "lr": LEARNING_RATE,
    "logging_steps": LOGGING_STEPS,
    "save_steps": SAVE_STEPS,
    "warmup_steps": WARMUP_STEPS,
})

{'batch_size': 2, 'grad_accum_steps': 4, 'epochs': 2, 'lr': 0.0002, 'logging_steps': 10, 'save_steps': 100, 'warmup_steps': 10}


In [40]:
if FastLanguageModel is None or "model" not in globals() or "tokenizer" not in globals():
    print("Trainer setup skipped locally. Run on HPC after model/tokenizer load.")
else:
    training_args = TrainingArguments(
        output_dir=str(SFT_OUTPUT_DIR),
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        warmup_steps=WARMUP_STEPS,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
        remove_unused_columns=False,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset_text,
        dataset_text_field="text",
        args=training_args,
    )

    print("Trainer initialized successfully.")

Trainer setup skipped locally. Run on HPC after model/tokenizer load.


In [41]:
if "trainer" not in globals():
    print("Training skipped locally. Run this on HPC.")
else:
    trainer.train()

Training skipped locally. Run this on HPC.
